### 1. Métodos avançados de dados em painel

O conjunto de códigos abaixo replica alguns exemplos práticos do Capítulo 14, intitulado "Métodos avançados de dados em painel", do livro Introdução à Econometria, tradução da 6º edição Norte Americana, do autor Jeffrey M. Wooldridge. Este capítulo aborda métodos um pouco mais avançados de dados em painel, como o estimador de efeitos fixos e a abordagem de efeitos aleatórios. O livro, em formato digital e físico, pode ser adquirido no seguinte site: [amazon.com.br](https://www.amazon.com.br/Introdu%C3%A7%C3%A3o-%C3%A0-econometria-abordagem-moderna/dp/8522125643?ref_=ast_author_dp&dib=eyJ2IjoiMSJ9.xpLzGp_gZEtZm2kFmBCS0if6iw5aeB-m4qX1SKcoLrzW2xj1vkpOLp1CDoM7qPymZwglN1USzM69oUsMlZMlc8hm2kJEe2SLc4ft_i_GErH5YT2BZepNJLVH2D1cAGzU3WfbeyFYrMs4djQeG0V0I3MA2aVgo4IqjdGSKZzvuHkfd18qWqoBjSYovRqPNAU0sI9kwF6RwGG2EatJQ0LuUUOSw_OTKr29SeqG01Y7cds.zp4BJt5PaAnnWcFufRwtdk6xs-2WFdGq2UM5uH3Xrbc&dib_tag=AUTHOR).

### 2. Bibliotecas

In [69]:
# Manipulação dos dados
import pandas as pd

# Estatística
from linearmodels.panel import PanelOLS

# Dados
import wooldridge as woo

# Configurações
import warnings

In [70]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
warnings.filterwarnings("ignore")

### Exemplo 14.1 - Efeito do treinamento de pessoal sobre as taxas de refugos de produtos das empresas

Estimando um modelo de efeitos não observados por efeitos fixos.

In [71]:
# Importando os dados
jtrain = woo.dataWoo("jtrain")
jtrain.head()

,year,fcode,employ,sales,avgsal,scrap,rework,tothrs,union,grant,d89,d88,totrain,hrsemp,lscrap,lemploy,lsales,lrework,lhrsemp,lscrap_1,grant_1,clscrap,cgrant,clemploy,clsales,lavgsal,clavgsal,cgrant_1,chrsemp,clhrsemp
0,1987,410032.0,100.0,47000000.0,35000.0,NaN,NaN,12.0,0,0,0,0,100.0,12.000000,NaN,4.605170,17.665659,NaN,2.564949,NaN,0,NaN,0,NaN,NaN,10.463103,NaN,NaN,NaN,NaN
1,1988,410032.0,131.0,43000000.0,37000.0,NaN,NaN,8.0,0,0,0,1,50.0,3.053435,NaN,4.875197,17.576710,NaN,1.399565,NaN,0,NaN,0,0.270027,-0.088949,10.518673,0.055570,0.0,-8.946565,-1.165385
2,1989,410032.0,123.0,49000000.0,39000.0,NaN,NaN,8.0,0,0,1,0,50.0,3.252033,NaN,4.812184,17.707331,NaN,1.447397,NaN,0,NaN,0,-0.063013,0.130621,10.571317,0.052644,0.0,0.198597,0.047832
3,1987,410440.0,12.0,1560000.0,10500.0,NaN,NaN,12.0,0,0,0,0,12.0,12.000000,NaN,2.484907,14.260197,NaN,2.564949,NaN,0,NaN,0,NaN,NaN,9.259130,NaN,NaN,NaN,NaN
4,1988,410440.0,13.0,1970000.0,11000.0,NaN,NaN,12.0,0,0,0,1,13.0,12.000000,NaN,2.564949,14.493544,NaN,2.564949,NaN,0,NaN,0,0.080043,0.233347,9.305651,0.046520,0.0,0.000000,0.000000


In [72]:
# Formato dos dados
jtrain.shape

(471, 30)

In [73]:
# Dados faltantes por coluna
#jtrain.isna().mean()

In [74]:
# Ajustando os índices (ano e identificador)
jtrain = jtrain.set_index(['fcode', 'year'])

In [75]:
jtrain.head()

employ       sales   avgsal  scrap  rework  tothrs  union  \
fcode    year                                                              
410032.0 1987   100.0  47000000.0  35000.0    NaN     NaN    12.0      0   
         1988   131.0  43000000.0  37000.0    NaN     NaN     8.0      0   
         1989   123.0  49000000.0  39000.0    NaN     NaN     8.0      0   
410440.0 1987    12.0   1560000.0  10500.0    NaN     NaN    12.0      0   
         1988    13.0   1970000.0  11000.0    NaN     NaN    12.0      0   

               grant  d89  d88  totrain     hrsemp  lscrap   lemploy  \
fcode    year                                                          
410032.0 1987      0    0    0    100.0  12.000000     NaN  4.605170   
         1988      0    0    1     50.0   3.053435     NaN  4.875197   
         1989      0    1    0     50.0   3.252033     NaN  4.812184   
410440.0 1987      0    0    0     12.0  12.000000     NaN  2.484907   
         1988      0    0    1     13.0  12.000000     NaN  2.564949   

                  lsales  lrework   lhrsemp  lscrap_1  grant_1  clscrap  \
fcode    year                                                             
410032.0 1987  17.665659      NaN  2.564949       NaN        0      NaN   
         1988  17.576710      NaN  1.399565       NaN        0      NaN   
         1989  17.707331      NaN  1.447397       NaN        0      NaN   
410440.0 1987  14.260197      NaN  2.564949       NaN        0      NaN   
         1988  14.493544      NaN  2.564949       NaN        0      NaN   

               cgrant  clemploy   clsales    lavgsal  clavgsal  cgrant_1  \
fcode    year                                                              
410032.0 1987       0       NaN       NaN  10.463103       NaN       NaN   
         1988       0  0.270027 -0.088949  10.518673  0.055570       0.0   
         1989       0 -0.063013  0.130621  10.571317  0.052644       0.0   
410440.0 1987       0       NaN       NaN   9.259130       NaN       NaN   
         1988       0  0.080043  0.233347   9.305651  0.046520       0.0   

                chrsemp  clhrsemp  
fcode    year                      
410032.0 1987       NaN       NaN  
         1988 -8.946565 -1.165385  
         1989  0.198597  0.047832  
410440.0 1987       NaN       NaN  
         1988  0.000000  0.000000

In [76]:
# Regressão
model = PanelOLS.from_formula('lscrap ~ d88 + d89 + grant + grant_1 + EntityEffects', data=jtrain)
results = model.fit()

# Exibindo resultados
print(results.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                 lscrap   R-squared:                        0.2010
Estimator:                   PanelOLS   R-squared (Between):             -0.1103
No. Observations:                 162   R-squared (Within):               0.2010
Date:                Mon, Jan 27 2025   R-squared (Overall):             -0.0839
Time:                        16:51:32   Log-likelihood                   -80.946
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      6.5426
Entities:                          54   P-value                           0.0001
Avg Obs:                       3.0000   Distribution:                   F(4,104)
Min Obs:                       3.0000                                           
Max Obs:                       3.0000   F-statistic (robust):             6.5426
                            

### Exemplo 14.2 - O retorno da educação mudou no transcorrer do tempo ?

In [77]:
# Importando os dados
wagepan = woo.dataWoo("wagepan")
wagepan.head()

,nr,year,agric,black,bus,construc,ent,exper,fin,hisp,poorhlth,hours,manuf,married,min,nrthcen,nrtheast,occ1,occ2,occ3,occ4,occ5,occ6,occ7,occ8,occ9,per,pro,pub,rur,south,educ,tra,trad,union,lwage,d81,d82,d83,d84,d85,d86,d87,expersq
0,13,1980,0,0,1,0,0,1,0,0,0,2672,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,14,0,0,0,1.197540,0,0,0,0,0,0,0,1
1,13,1981,0,0,0,0,0,2,0,0,0,2320,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,14,0,0,1,1.853060,1,0,0,0,0,0,0,4
2,13,1982,0,0,1,0,0,3,0,0,0,2940,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,14,0,0,0,1.344462,0,1,0,0,0,0,0,9
3,13,1983,0,0,1,0,0,4,0,0,0,2960,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,14,0,0,0,1.433213,0,0,1,0,0,0,0,16
4,13,1984,0,0,0,0,0,5,0,0,0,3071,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,14,0,0,0,1.568125,0,0,0,1,0,0,0,25


In [78]:
# Formato dos dados
wagepan.shape

(4360, 44)

In [79]:
# Dados faltantes
# wagepan.isna().sum()

In [80]:
# Ajustando os índices (ano e identificador)
wagepan = wagepan.set_index(['nr', 'year'], drop=False)

In [81]:
wagepan.head()

nr  year  agric  black  bus  construc  ent  exper  fin  hisp  \
nr year                                                                 
13 1980  13  1980      0      0    1         0    0      1    0     0   
   1981  13  1981      0      0    0         0    0      2    0     0   
   1982  13  1982      0      0    1         0    0      3    0     0   
   1983  13  1983      0      0    1         0    0      4    0     0   
   1984  13  1984      0      0    0         0    0      5    0     0   

         poorhlth  hours  manuf  married  min  nrthcen  nrtheast  occ1  occ2  \
nr year                                                                        
13 1980         0   2672      0        0    0        0         1     0     0   
   1981         0   2320      0        0    0        0         1     0     0   
   1982         0   2940      0        0    0        0         1     0     0   
   1983         0   2960      0        0    0        0         1     0     0   
   1984         0   3071      0        0    0        0         1     0     0   

         occ3  occ4  occ5  occ6  occ7  occ8  occ9  per  pro  pub  rur  south  \
nr year                                                                        
13 1980     0     0     0     0     0     0     1    0    0    0    0      0   
   1981     0     0     0     0     0     0     1    1    0    0    0      0   
   1982     0     0     0     0     0     0     1    0    0    0    0      0   
   1983     0     0     0     0     0     0     1    0    0    0    0      0   
   1984     0     0     1     0     0     0     0    1    0    0    0      0   

         educ  tra  trad  union     lwage  d81  d82  d83  d84  d85  d86  d87  \
nr year                                                                        
13 1980    14    0     0      0  1.197540    0    0    0    0    0    0    0   
   1981    14    0     0      1  1.853060    1    0    0    0    0    0    0   
   1982    14    0     0      0  1.344462    0    1    0    0    0    0    0   
   1983    14    0     0      0  1.433213    0    0    1    0    0    0    0   
   1984    14    0     0      0  1.568125    0    0    0    1    0    0    0   

         expersq  
nr year           
13 1980        1  
   1981        4  
   1982        9  
   1983       16  
   1984       25

In [82]:
# Regressão
model = PanelOLS.from_formula('lwage ~ married + union + C(year)*educ + EntityEffects', data=wagepan, drop_absorbed=True)
results = model.fit()

# Exibindo resultados
print(results.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                        0.1708
Estimator:                   PanelOLS   R-squared (Between):              0.0905
No. Observations:                4360   R-squared (Within):               0.1708
Date:                Mon, Jan 27 2025   R-squared (Overall):              0.1277
Time:                        16:51:38   Log-likelihood                   -1350.7
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      48.907
Entities:                         545   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                 F(16,3799)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             5454.7
                            

### Exemplo 14.4 - Uma equação de Salários com dados em painel

In [83]:
# Importando os dados
wagepan = woo.dataWoo("wagepan")
wagepan.head()

,nr,year,agric,black,bus,construc,ent,exper,fin,hisp,poorhlth,hours,manuf,married,min,nrthcen,nrtheast,occ1,occ2,occ3,occ4,occ5,occ6,occ7,occ8,occ9,per,pro,pub,rur,south,educ,tra,trad,union,lwage,d81,d82,d83,d84,d85,d86,d87,expersq
0,13,1980,0,0,1,0,0,1,0,0,0,2672,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,14,0,0,0,1.197540,0,0,0,0,0,0,0,1
1,13,1981,0,0,0,0,0,2,0,0,0,2320,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,14,0,0,1,1.853060,1,0,0,0,0,0,0,4
2,13,1982,0,0,1,0,0,3,0,0,0,2940,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,14,0,0,0,1.344462,0,1,0,0,0,0,0,9
3,13,1983,0,0,1,0,0,4,0,0,0,2960,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,14,0,0,0,1.433213,0,0,1,0,0,0,0,16
4,13,1984,0,0,0,0,0,5,0,0,0,3071,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,14,0,0,0,1.568125,0,0,0,1,0,0,0,25


In [84]:
# Ajustando os índices (ano e identificador)
wagepan = wagepan.set_index(['nr', 'year'], drop=False)

##### MQO Agrupado

In [85]:
model_pooled = plm.PooledOLS.from_formula(formula='lwage ~ educ + black + hisp + exper + expersq + married + union + C(year)', data=wagepan)
results_pooled = model_pooled.fit()

# Exibindo resultados
print(results_pooled.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                        0.1893
Estimator:                  PooledOLS   R-squared (Between):              0.2066
No. Observations:                4360   R-squared (Within):               0.1692
Date:                Mon, Jan 27 2025   R-squared (Overall):              0.1893
Time:                        16:51:42   Log-likelihood                   -2982.0
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      72.459
Entities:                         545   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                 F(14,4345)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             3381.6
                            

##### Efeitos aleatórios

In [86]:
model_re = plm.RandomEffects.from_formula(formula='lwage ~ educ + black + hisp + exper + expersq + married + union + C(year)', data=wagepan)
results_re = model_re.fit()

# Exibindo resultados
print(results_re.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                  lwage   R-squared:                        0.1806
Estimator:              RandomEffects   R-squared (Between):              0.1853
No. Observations:                4360   R-squared (Within):               0.1799
Date:                Mon, Jan 27 2025   R-squared (Overall):              0.1828
Time:                        16:51:50   Log-likelihood                   -1622.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      68.409
Entities:                         545   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                 F(14,4345)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             846.19
                            

##### Efeitos fixos

In [87]:
model_fe = plm.PanelOLS.from_formula(formula='lwage ~ expersq + married + union + C(year) + EntityEffects', data=wagepan)
results_fe = model_fe.fit()

# Exibindo resultados
print(results_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  lwage   R-squared:                        0.1806
Estimator:                   PanelOLS   R-squared (Between):             -0.0052
No. Observations:                4360   R-squared (Within):               0.1806
Date:                Mon, Jan 27 2025   R-squared (Overall):              0.0807
Time:                        16:51:50   Log-likelihood                   -1324.8
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      83.851
Entities:                         545   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                 F(10,3805)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             8850.2
                            